<a href="https://colab.research.google.com/github/salma-zr/Deep-Learning/blob/cursor%2Fcompl-tude-de-t-che-8adb/compl-tude-de-t-che-8adb/Medical_QA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 Medical Question Answering - Full Pipeline

**Projet Deep Learning - Clé en main**

Ce notebook exécute automatiquement:
1. Installation des dépendances
2. Téléchargement du dataset
3. Expériences (Flan-T5, OpenAI si clé fournie)
4. Évaluation (ROUGE, BLEU)
5. Génération du rapport PDF

⚠️ **Medical Disclaimer**: Projet éducatif uniquement. Ne pas utiliser pour des conseils médicaux.

## 1️⃣ Configuration (EXÉCUTER EN PREMIER)

In [1]:
#@title ⚙️ Configuration
#@markdown ### Clés API (optionnel mais recommandé)
OPENAI_API_KEY = ""  #@param {type:"string"}
#@markdown ### Mode d'exécution
MODE = "quick"  #@param ["quick", "full"]
#@markdown - **quick**: 50 exemples, ~10 min
#@markdown - **full**: 500 exemples, ~1h

import os
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("✅ Clé OpenAI configurée")
else:
    print("⚠️ Pas de clé OpenAI - seules les expériences locales seront exécutées")

QUICK_MODE = MODE == "quick"
print(f"Mode: {MODE}")

⚠️ Pas de clé OpenAI - seules les expériences locales seront exécutées
Mode: quick


## 2️⃣ Installation

In [2]:
#@title 📦 Cloner le repo et installer les dépendances
%%time

import os

# Clone repo
if not os.path.exists("Deep-Learning"):
    !git clone https://github.com/salma-zr/Deep-Learning.git
    %cd Deep-Learning
    !git checkout cursor/compl-tude-de-t-che-8adb
else:
    %cd Deep-Learning
    !git pull origin cursor/compl-tude-de-t-che-8adb

# Install dependencies
!pip install -q -e .

print("\n✅ Installation terminée!")

Cloning into 'Deep-Learning'...
remote: Enumerating objects: 209, done.
remote: Counting objects: 100% (209/209), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 209 (delta 49), reused 200 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (209/209), 12.62 MiB | 14.80 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/Deep-Learning
Branch 'cursor/compl-tude-de-t-che-8adb' set up to track remote branch 'cursor/compl-tude-de-t-che-8adb' from 'origin'.
Switched to a new branch 'cursor/compl-tude-de-t-che-8adb'
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 3️⃣ Préparation des données

In [3]:
#@title 📊 Télécharger le dataset et créer les splits
%%time

!python -m src.data.cli download
!python -m src.data.cli split --seed 42 --tiny-size 200
!python -m src.data.cli info

print("\n✅ Dataset prêt!")

[01/27/26 16:12:23] INFO     Downloading dataset:                               
                             medalpaca/medical_meadow_medical_flashcards        
README.md: 1.24kB [00:00, 5.36MB/s]
medical_meadow_wikidoc_medical_flashcard(…): 100% 17.7M/17.7M [00:00<00:00, 21.3MB/s]
Generating train split: 100% 33955/33955 [00:00<00:00, 100091.57 examples/s]
[01/27/26 16:12:26] INFO     Downloaded 33955 examples                          
[01/27/26 16:12:28] INFO     Saved dataset to                                   
                             /content/Deep-Learning/data/raw/medical_flashcards.
                             jsonl                                              
                    INFO     Question length: mean=92 chars                     
                    INFO     Answer length: mean=349 chars                      
Dataset saved to: /content/Deep-Learning/data/raw/medical_flashcards.jsonl
Creating dataset splits...
Ratios: train=0.8, dev=0.1, test=0.1
Seed: 42
[01/2

## 4️⃣ Expériences

In [4]:
#@title 🧪 Créer les dossiers de résultats
!mkdir -p results/preds results/scores results/qualitative results/figures
!mkdir -p report/tables report/figures

In [5]:
#@title 🤖 Expérience 1: Flan-T5 (CPU, gratuit)
%%time

split = "tiny_test" if QUICK_MODE else "test"
limit = 50 if QUICK_MODE else 500

!python -m src.generation.cli run configs/exp_11_hf_flan_t5.yaml --split {split} --limit {limit}
print("\n✅ Expérience Flan-T5 terminée!")

[01/27/26 16:12:37] INFO     ===================================================
                    INFO     EXPERIMENT: exp_11_hf_flan_t5                      
                    INFO     Started at: 2026-01-27T16:12:37.055821             
                    INFO     ---------------------------------------------------
                             ---------                                          
                    INFO     Configuration:                                     
                    INFO       name: exp_11_hf_flan_t5                          
                    INFO       strategy: closed_book                            
                    INFO       description: Flan-T5-base via HuggingFace        
                             (CPU-friendly fallback)                            
                    INFO       backend: huggingface                             
                    INFO       model: google/flan-t5-base                       
                    INFO    

In [6]:
#@title 🤖 Expérience 2: Flan-T5 + Flashcard Prompt
%%time

!python -m src.generation.cli run configs/exp_12_hf_flashcard.yaml --split {split} --limit {limit}
print("\n✅ Expérience Flashcard terminée!")

[01/27/26 16:14:11] INFO     ===================================================
                    INFO     EXPERIMENT: exp_12_hf_flashcard                    
                    INFO     Started at: 2026-01-27T16:14:11.959048             
                    INFO     ---------------------------------------------------
                             ---------                                          
                    INFO     Configuration:                                     
                    INFO       name: exp_12_hf_flashcard                        
                    INFO       strategy: closed_book                            
                    INFO       description: Flan-T5-base with flashcard-style   
                             prompt                                             
                    INFO       backend: huggingface                             
                    INFO       model: google/flan-t5-base                       
                    INFO    

In [7]:
#@title 🤖 Expérience 3: Flan-T5 + One-Sentence Prompt
%%time

!python -m src.generation.cli run configs/exp_13_hf_one_sentence.yaml --split {split} --limit {limit}
print("\n✅ Expérience One-Sentence terminée!")

[01/27/26 16:15:33] INFO     ===================================================
                    INFO     EXPERIMENT: exp_13_hf_one_sentence                 
                    INFO     Started at: 2026-01-27T16:15:33.265646             
                    INFO     ---------------------------------------------------
                             ---------                                          
                    INFO     Configuration:                                     
                    INFO       name: exp_13_hf_one_sentence                     
                    INFO       strategy: closed_book                            
                    INFO       description: Flan-T5-base with                   
                             one-sentence-strict prompt                         
                    INFO       backend: huggingface                             
                    INFO       model: google/flan-t5-base                       
                    INFO    

In [8]:
#@title 🌐 Expérience 4: OpenAI GPT-4o-mini (nécessite clé API)
%%time

import os
if os.environ.get("OPENAI_API_KEY"):
    !python -m src.generation.cli run configs/exp_01_openai_baseline.yaml --split {split} --limit {limit}
    print("\n✅ Expérience OpenAI terminée!")
else:
    print("⏭️ Skipped - Pas de clé OpenAI")

⏭️ Skipped - Pas de clé OpenAI
CPU times: user 51 µs, sys: 7 µs, total: 58 µs
Wall time: 60.6 µs


In [9]:
#@title 🌐 Expérience 5: OpenAI + Flashcard Prompt
%%time

import os
if os.environ.get("OPENAI_API_KEY"):
    !python -m src.generation.cli run configs/exp_04_prompt_flashcard.yaml --split {split} --limit {limit}
    print("\n✅ Expérience OpenAI Flashcard terminée!")
else:
    print("⏭️ Skipped - Pas de clé OpenAI")

⏭️ Skipped - Pas de clé OpenAI
CPU times: user 43 µs, sys: 6 µs, total: 49 µs
Wall time: 52.5 µs


In [10]:
#@title 🌐 Expérience 6: OpenAI + Uncertainty Prompt
%%time

import os
if os.environ.get("OPENAI_API_KEY"):
    !python -m src.generation.cli run configs/exp_05_prompt_uncertainty.yaml --split {split} --limit {limit}
    print("\n✅ Expérience OpenAI Uncertainty terminée!")
else:
    print("⏭️ Skipped - Pas de clé OpenAI")

⏭️ Skipped - Pas de clé OpenAI
CPU times: user 54 µs, sys: 8 µs, total: 62 µs
Wall time: 85.8 µs


In [11]:
#@title 📚 Expérience 7: RAG Wikipedia (nécessite clé API)
%%time

import os
if os.environ.get("OPENAI_API_KEY"):
    !python -m src.rag.cli run configs/exp_06_rag_wikipedia.yaml --split {split} --limit {limit}
    print("\n✅ Expérience RAG Wikipedia terminée!")
else:
    print("⏭️ Skipped - Pas de clé OpenAI")

⏭️ Skipped - Pas de clé OpenAI
CPU times: user 0 ns, sys: 57 µs, total: 57 µs
Wall time: 59.8 µs


## 5️⃣ Évaluation

In [12]:
#@title 📈 Calculer les métriques pour toutes les expériences
%%time

import glob

pred_files = glob.glob("results/preds/*.jsonl")
print(f"Fichiers de prédictions trouvés: {len(pred_files)}")

for pred_file in pred_files:
    if "_judged" in pred_file:
        continue
    name = pred_file.split("/")[-1].replace(".jsonl", "")
    print(f"\n📊 Évaluation de {name}...")
    !python -m src.eval.cli metrics "{pred_file}" --output "results/scores/{name}.csv"

print("\n✅ Évaluation terminée!")

Fichiers de prédictions trouvés: 3

📊 Évaluation de exp_11_hf_flan_t5...
Computing metrics for results/preds/exp_11_hf_flan_t5.jsonl...
          Evaluation Metrics          
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Metric                ┃ Value      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ avg_length_chars      │ 25.6531    │
│ avg_length_words      │ 4.2245     │
│ bleu                  │ 0.0000     │
│ empty_pct             │ 2.0000     │
│ insufficient_info_pct │ 0.0000     │
│ latency_max_ms        │ 17040.8307 │
│ latency_mean_ms       │ 1102.5676  │
│ latency_median_ms     │ 615.6027   │
│ latency_min_ms        │ 269.5096   │
│ latency_p95_ms        │ 1888.4805  │
│ multi_sentence_pct    │ 0.0000     │
│ n_examples            │ 50         │
│ rouge1                │ 0.1262     │
│ rouge2                │ 0.0665     │
│ rougeL                │ 0.1189     │
└───────────────────────┴────────────┘
Metrics saved to results/scores/exp_11_hf_flan_t5.csv

📊 Évaluation de exp_12_hf_fla

In [13]:
#@title 🧑‍⚖️ LLM Judge (nécessite clé API)
%%time

import os
import glob

if os.environ.get("OPENAI_API_KEY"):
    pred_files = glob.glob("results/preds/*.jsonl")
    for pred_file in pred_files:
        if "_judged" in pred_file:
            continue
        name = pred_file.split("/")[-1].replace(".jsonl", "")
        print(f"\n🧑‍⚖️ Judging {name}...")
        !python -m src.eval.cli judge "{pred_file}" --judge-model gpt-4o-mini --limit 50
    print("\n✅ Jugement terminé!")
else:
    print("⏭️ Skipped - Pas de clé OpenAI pour le LLM Judge")

⏭️ Skipped - Pas de clé OpenAI pour le LLM Judge
CPU times: user 56 µs, sys: 0 ns, total: 56 µs
Wall time: 58.4 µs


## 6️⃣ Génération du Rapport

In [14]:
#@title 📊 Générer les tableaux et figures
%%time

!python -m src.report.cli tables
!python -m src.report.cli figures

print("\n✅ Tableaux et figures générés!")

Generating LaTeX tables...
[01/27/26 16:16:42] INFO     Results table saved to                             
                             /content/Deep-Learning/report/tables/main_results.t
                             ex                                                 
Main results table: /content/Deep-Learning/report/tables/main_results.tex
Ablation table: /content/Deep-Learning/report/tables/ablation.tex
Tables generated successfully!
Generating figures...
[01/27/26 16:16:45] INFO     Metrics comparison plot saved to                   
                             /content/Deep-Learning/report/figures/metrics_compa
                             rison.png                                          
Generated: /content/Deep-Learning/report/figures/metrics_comparison.png
Generated: /content/Deep-Learning/report/figures/judge_distribution.png
Generated: /content/Deep-Learning/report/figures/ablation_curve.png
Figures generated successfully!

✅ Tableaux et figures générés!
CPU times: user 17

In [ ]:
#@title 📄 Installer LaTeX et compiler le PDF
%%time

# Install LaTeX
!apt-get update -qq && apt-get install -qq -y texlive-latex-base texlive-latex-extra > /dev/null 2>&1

# Compile PDF
!python -m src.report.cli compile report/report.tex

print("\n✅ PDF compilé!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
#@title 📥 Télécharger le rapport PDF
from google.colab import files

# Download PDF
files.download('report/report.pdf')
print("\n✅ Téléchargement lancé!")

## 7️⃣ Visualisation des Résultats

In [ ]:
#@title 📊 Afficher le tableau des résultats
import pandas as pd
import glob

# Load all score files
score_files = glob.glob("results/scores/*.csv")
if score_files:
    dfs = [pd.read_csv(f) for f in score_files]
    results = pd.concat(dfs, ignore_index=True)

    # Display key metrics
    cols = ['experiment', 'rougeL', 'bleu', 'latency_mean_ms', 'n_examples']
    cols = [c for c in cols if c in results.columns]
    display(results[cols].sort_values('rougeL', ascending=False))
else:
    print("Aucun fichier de scores trouvé")

In [ ]:
#@title 📈 Afficher les figures générées
from IPython.display import Image, display
import os

figures = [
    "report/figures/metrics_comparison.png",
    "report/figures/judge_distribution.png",
    "report/figures/ablation_curve.png"
]

for fig in figures:
    if os.path.exists(fig):
        print(f"\n📊 {fig}")
        display(Image(fig, width=600))

In [ ]:
#@title 🔍 Exemples de prédictions
import json
import glob

pred_files = glob.glob("results/preds/*.jsonl")
if pred_files:
    # Show examples from first file
    with open(pred_files[0]) as f:
        examples = [json.loads(line) for line in f][:5]

    print(f"📂 Fichier: {pred_files[0]}\n")
    for i, ex in enumerate(examples, 1):
        print(f"--- Exemple {i} ---")
        print(f"❓ Question: {ex['question'][:100]}...")
        print(f"✅ Référence: {ex['reference'][:100]}...")
        print(f"🤖 Prédiction: {ex['prediction']}")
        print()

## 8️⃣ Fine-tuning (Optionnel, GPU requis)

In [ ]:
#@title 🔧 Vérifier GPU disponible
import torch

if torch.cuda.is_available():
    print(f"✅ GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"   Mémoire: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ Pas de GPU - Le fine-tuning sera très lent ou impossible")
    print("   → Va dans Runtime > Change runtime type > T4 GPU")

In [ ]:
#@title 📚 Préparer les données de fine-tuning
%%time

!python -m src.finetune.cli prepare --max 1000 --style alpaca
print("\n✅ Données préparées!")

In [ ]:
#@title 🚀 Lancer le fine-tuning (1k exemples)
%%time

import torch

if torch.cuda.is_available():
    !python -m src.finetune.cli train configs/exp_08_finetune_1k.yaml
    print("\n✅ Fine-tuning terminé!")
else:
    print("⏭️ Skipped - Pas de GPU disponible")
    print("   Exécution en mode symbolique pour test...")
    !python -m src.finetune.cli train configs/exp_08_finetune_1k.yaml --symbolic

## 📥 Télécharger tous les résultats

In [ ]:
#@title 📦 Créer une archive ZIP de tous les résultats
!zip -r medical_qa_results.zip results/ report/report.pdf report/tables/ report/figures/

from google.colab import files
files.download('medical_qa_results.zip')
print("\n✅ Archive téléchargée!")

---
## ✅ Checklist Finale

Avant de soumettre, vérifie que:

- [ ] Le PDF `report/report.pdf` est généré
- [ ] Les tableaux montrent des résultats réels
- [ ] Les figures sont lisibles
- [ ] Tu as personnalisé la section "Qualitative Analysis" dans le rapport
- [ ] Tu as relu la section "Discussion & Limitations"

**Bon courage !** 🎓